In [ ]:
#!/usr/bin/env python3
"""
fix_columns.py

Uso:
    python fix_columns.py caminho/do/arquivo.csv
    python fix_columns.py caminho/do/arquivo.xlsx

Resultado:
    Gera: caminho/do/arquivo_formatted.xlsx
"""

import sys
import os
import csv
from io import StringIO
import pandas as pd

def detect_delimiter(sample):
    """Tenta detectar delimitador via csv.Sniffer, senão usa heurística por contagem."""
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=[',',';','\t','|'])
        return dialect.delimiter
    except Exception:
        counts = {d: sample.count(d) for d in [',',';','\t','|']}
        # retorna o delimitador com maior ocorrência (padrão ',')
        return max(counts, key=counts.get)

def parse_csv_line(line, delim):
    """Parseia uma linha CSV (respeitando aspas) e retorna lista de campos."""
    return next(csv.reader([line], delimiter=delim, quotechar='"'))

def fix_single_column_df(df, delim=None):
    """
    Quando df tem apenas 1 coluna onde cada célula é uma linha CSV completa,
    parseia cada célula via csv.reader para obter colunas reais.
    Assume que a primeira linha contém os headers.
    """
    col0 = df.iloc[:,0].astype(str).tolist()
    sample = '\n'.join(col0[:10])
    if delim is None:
        delim = detect_delimiter(sample)
    parsed = [parse_csv_line(line, delim) for line in col0]
    parsed_df = pd.DataFrame(parsed)
    # Usa a primeira linha como header
    parsed_df.columns = parsed_df.iloc[0].astype(str)
    parsed_df = parsed_df[1:].reset_index(drop=True)
    # limpar aspas sobrando
    parsed_df.columns = [c.strip().replace('"','') for c in parsed_df.columns]
    parsed_df = parsed_df.applymap(lambda x: str(x).strip().replace('"','') if pd.notnull(x) else x)
    return parsed_df, delim

def main(infile, outfile=None, encoding='utf-8'):
    if not os.path.isfile(infile):
        print("Arquivo não encontrado:", infile)
        return

    base, ext = os.path.splitext(infile)
    if outfile is None:
        outfile = f"{base}_formatted.xlsx"

    ext = ext.lower()
    # 1) Ler arquivo conforme extensão
    if ext in ('.xls', '.xlsx'):
        # lê tudo sem interpretar header (header=None) para evitar perda de dados
        df = pd.read_excel(infile, header=None, dtype=str)
    else:
        # tenta detectar delimitador a partir das primeiras linhas
        with open(infile, 'r', encoding=encoding, errors='replace') as f:
            sample_lines = ''.join([f.readline() for _ in range(10)])
        delim = detect_delimiter(sample_lines)
        try:
            # tenta ler 'normalmente' com pandas
            df = pd.read_csv(infile, sep=delim, quotechar='"', encoding=encoding, dtype=str, engine='python')
        except Exception:
            # fallback: ler como uma única coluna de linhas brutas
            with open(infile, 'r', encoding=encoding, errors='replace') as f:
                lines = [line.rstrip('\n\r') for line in f]
            df = pd.DataFrame(lines)

    # 2) Se o dataframe resultante tem apenas 1 coluna, provavelmente cada célula é uma linha CSV
    if df.shape[1] == 1:
        fixed_df, used_delim = fix_single_column_df(df)
        fixed_df.to_excel(outfile, index=False)
        print("Arquivo salvo em:", outfile)
        print("Delimitador detectado e usado:", repr(used_delim))
        print("Colunas:", list(fixed_df.columns))
        print("Linhas:", fixed_df.shape[0])
        return

    # 3) Caso já esteja multi-coluna, só limpar aspas e salvar
    df.columns = [str(c).strip().replace('"','') for c in df.columns]
    df = df.applymap(lambda x: str(x).strip().replace('"','') if pd.notnull(x) else x)
    df.to_excel(outfile, index=False)
    print("Arquivo salvo em:", outfile)
    print("Colunas:", list(df.columns))
    print("Linhas:", df.shape[0])

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Uso: python fix_columns.py caminho/do/arquivo.csv (ou .xlsx)")
    else:
        main(sys.argv[1])
